# Sales Forecasting — Exploratory Data Analysis

Quick EDA on the state-level weekly Beverages sales dataset.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.preprocessing import load_raw, resample_weekly, add_features, get_state_data

plt.rcParams['figure.figsize'] = (14, 5)
sns.set_style('whitegrid')

In [ ]:
raw = load_raw()
print('Raw shape:', raw.shape)
raw.head()

In [ ]:
print('States:', raw['State'].nunique())
print('Date range:', raw['Date'].min(), '→', raw['Date'].max())
print('Missing values:\n', raw.isnull().sum())

In [ ]:
weekly = resample_weekly(raw)
print('Weekly shape:', weekly.shape)
# Check date regularity for one state
sample = weekly[weekly['State']=='California'].sort_values('Date')
diffs = sample['Date'].diff().dropna().dt.days.value_counts()
print('Date gaps (California):\n', diffs)

In [ ]:
# Plot top 6 states by total sales
top6 = weekly.groupby('State')['Total'].sum().nlargest(6).index.tolist()
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, state in zip(axes.flat, top6):
    s = weekly[weekly['State']==state].sort_values('Date')
    ax.plot(s['Date'], s['Total'] / 1e6)
    ax.set_title(state)
    ax.set_ylabel('Sales ($M)')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('eda_top6_states.png', dpi=100)
plt.show()

In [ ]:
# Seasonality: average sales by month (all states)
weekly['month'] = pd.to_datetime(weekly['Date']).dt.month
monthly_avg = weekly.groupby('month')['Total'].mean()
monthly_avg.plot(kind='bar', color='steelblue')
plt.title('Average Weekly Sales by Month (all states)')
plt.xlabel('Month')
plt.ylabel('Avg Sales ($)')
plt.tight_layout()
plt.savefig('eda_monthly_seasonality.png', dpi=100)
plt.show()

In [ ]:
# Feature engineering preview
featured = add_features(weekly)
state_data = get_state_data(featured)
ca = state_data['California']
print(ca[['Date','Total','lag_1','lag_4','lag_52','roll_mean_12','holiday_flag']].tail(10))

In [ ]:
# Correlation heatmap of features (California)
feat_cols = ['Total','lag_1','lag_4','lag_52','roll_mean_4','roll_mean_12','month','quarter','holiday_flag']
corr = ca[feat_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation — California')
plt.tight_layout()
plt.savefig('eda_feature_correlation.png', dpi=100)
plt.show()